# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and processing the [FAIR<sup>2</sup> dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2) using the `mlcroissant` library and referencing all entities via their Croissant `@id` fields.

### Dataset Source
The dataset is described via a Croissant schema JSON-LD file, accessible at:

- [`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. This will allow us to inspect the structure and content of the dataset using Croissant's semantics.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display high level metadata
print("Name:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Published:", dataset.metadata.datePublished)
print("License:", dataset.metadata.license)

# List Croissant @id for the dataset
print("Dataset @id:", dataset.metadata['@id'])

## 2. Data Overview

Let's review available RecordSets in the dataset and fields they contain, referencing each by their Croissant `@id`.


A RecordSet roughly maps to a table, and each field is a column. We'll print all available RecordSet `@id`s and for each, list their field `@id`s.

In [ ]:
# Retrieve and list all RecordSets in the dataset
record_sets = list(dataset.metadata.recordSets)
print(f"Found {len(record_sets)} record sets.\n")

for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs['@id']}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field['@id']}, type: {field.dataType})")
    print()

## 3. Data Extraction

Let's extract data from a specific RecordSet into a pandas DataFrame. We will use `@id` for referencing everything.

In this dataset, the principal RecordSet most likely contains the mass spec proteins table. Here we load all tabular record sets as DataFrames, indexed by their `@id`.

In [ ]:
# Map from RecordSet @id to DataFrame
dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSets]

for rsid in record_set_ids:
    print(f"Loading records from RecordSet: {rsid}")
    records = list(dataset.records(record_set=rsid))
    if len(records) == 0:
        print(f"  - No records found for {rsid}. Skipping.")
        continue
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"  - Columns: {df.columns.tolist()}")
    print(f"  - Number of records: {len(df)}\n")

# Choose the main RecordSet for further analysis (adjust @id as needed)
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
print(f"Main record set chosen for EDA: {main_record_set_id}")

if main_record_set_id:
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's examine some of the numeric and categorical fields for further processing. You should reference fields by their `@id` as established previously.

In [ ]:
# Example: Suppose the main table has numeric fields like 'molecular_weight' or 'abundance'.
# Replace these example IDs with actual @id strings you found above.

import numpy as np

# Identify a numeric field for demonstration
numeric_field_id = None
possible_numeric_fields = []
if main_record_set_id:
    # Try to auto-detect a suitable numeric column
    df = dataframes[main_record_set_id]
    for col in df.columns:
        # Find one with likely numeric dtype
        if pd.api.types.is_numeric_dtype(df[col]):
            possible_numeric_fields.append(col)
    if possible_numeric_fields:
        numeric_field_id = possible_numeric_fields[0]  # Just pick the first one
        print(f"Selected numeric field for EDA: {numeric_field_id}")

if numeric_field_id:
    # Filtering records above a numeric threshold
    threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 1
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Example grouping if another categorical field exists
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and (df[col].dtype == 'object' or pd.api.types.is_categorical_dtype(df[col])):
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("No numeric field found in the main record set for demonstration.")

## 5. Visualization

Let's visualize the distribution of a numeric field and the effect of grouping by a categorical field, referencing fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping exists
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

- We have loaded the FAIR<sup>2</sup> mass spectrometry dataset using `mlcroissant`,
- Inspected its metadata and structure via Croissant `@id` semantics,
- Loaded all available RecordSets as tables,
- Performed basic EDA and visualized selected features.

You can extend this notebook by referencing the `@id`s shown previously for more advanced analyses or visualizations tailored to your research question.